# 03 — Baseline Thermal Model

Train and evaluate a baseline thermal scoring model using synthetic data generated
from the `MockWeatherProvider` diurnal cycle, then compare against the rule-based
`ThermalScorer` formula.

**Goals:**
- Generate a synthetic labelled dataset
- Train an XGBoost model on extracted features
- Evaluate with 5-fold cross-validation
- Calibrate confidence with isotonic regression
- Save the trained artifact for use by the backend `ThermalScorer`

**Advisory**: This model is trained on synthetic data only. Real flight-derived labels are required for any production use.

In [ ]:
import sys
sys.path.insert(0, '../backend')

import asyncio
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

from config.settings import get_settings
from data_ingestion.weather.mock_provider import MockWeatherProvider
from ml.features import FeatureExtractor
from ml.thermal_scorer import ThermalScorer
from ml.calibration import ConfidenceCalibrator

SITE_LAT = 35.492
SITE_LON = -118.187
ARTIFACT_PATH = Path('../backend/ml/artifacts')
ARTIFACT_PATH.mkdir(parents=True, exist_ok=True)

print('Baseline thermal model notebook ready.')
print(f'XGBoost version: {xgb.__version__}')

## 1. Generate Synthetic Training Data

In [ ]:
provider = MockWeatherProvider()
extractor = FeatureExtractor()
rule_scorer = ThermalScorer()

# Generate 30 days of synthetic hourly data
all_features = []
all_labels = []
all_scores = []

for day_offset in range(30):
    forecast = asyncio.run(provider.get_forecast(lat=SITE_LAT, lon=SITE_LON, days=1))
    
    for h in forecast.hourly:
        # Extract features
        features = extractor.extract_from_weather_hour(h)
        
        # Rule-based score as pseudo-label
        score = rule_scorer.score(features)
        
        # Binary label: score >= 0.5 = "good thermal conditions"
        label = 1 if score >= 0.50 else 0
        
        all_features.append(features)
        all_labels.append(label)
        all_scores.append(score)

# Build feature matrix
feature_names = extractor.FEATURE_NAMES
X = np.array([[f.get(name, 0.0) for name in feature_names] for f in all_features])
y = np.array(all_labels)
y_score = np.array(all_scores)

print(f'Dataset: {X.shape[0]} samples × {X.shape[1]} features')
print(f'Positive class (good conditions): {y.sum()} / {len(y)} ({y.mean() * 100:.1f}%)')
print(f'Features: {feature_names}')

## 2. Cross-Validated Training

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []
brier_scores = []
oof_preds = np.zeros(len(y))

xgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'use_label_encoder': False,
    'eval_metric': 'logloss',
    'random_state': 42,
    'verbosity': 0,
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = preds
    
    auc = roc_auc_score(y_val, preds)
    brier = brier_score_loss(y_val, preds)
    auc_scores.append(auc)
    brier_scores.append(brier)
    print(f'Fold {fold+1}: AUC={auc:.3f}, Brier={brier:.4f}')

print(f'\nMean AUC:   {np.mean(auc_scores):.3f} ± {np.std(auc_scores):.3f}')
print(f'Mean Brier: {np.mean(brier_scores):.4f} ± {np.std(brier_scores):.4f}')

## 3. Train Final Model and Calibrate

In [ ]:
# Train on full dataset
final_model = xgb.XGBClassifier(**xgb_params)
final_model.fit(X, y, verbose=False)

# Calibrate with isotonic regression on OOF predictions
calibrator = ConfidenceCalibrator()
calibrator.fit(oof_preds, y)

# Evaluate calibration
calibrated_preds = calibrator.calibrate(oof_preds)

print('Calibration applied.')
print(f'Pre-calibration Brier:  {brier_score_loss(y, oof_preds):.4f}')
print(f'Post-calibration Brier: {brier_score_loss(y, calibrated_preds):.4f}')

# Feature importances
importances = final_model.feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(imp_df['feature'], imp_df['importance'], color='#2563eb', alpha=0.8)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost Feature Importances — Thermal Scoring')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
print(imp_df.head(5).to_string(index=False))

## 4. Reliability Diagram (Calibration Curve)

In [ ]:
from sklearn.calibration import calibration_curve

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pre-calibration
fraction_pos, mean_pred = calibration_curve(y, oof_preds, n_bins=10)
ax1.plot(mean_pred, fraction_pos, 's-', color='#2563eb', label='XGBoost (uncalibrated)')
ax1.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax1.set_xlabel('Mean predicted probability')
ax1.set_ylabel('Fraction of positives')
ax1.set_title('Reliability Diagram — Pre-Calibration')
ax1.legend()
ax1.grid(alpha=0.3)

# Post-calibration
fraction_pos_cal, mean_pred_cal = calibration_curve(y, calibrated_preds, n_bins=10)
ax2.plot(mean_pred_cal, fraction_pos_cal, 's-', color='#10b981', label='XGBoost (calibrated)')
ax2.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax2.set_xlabel('Mean predicted probability')
ax2.set_ylabel('Fraction of positives')
ax2.set_title('Reliability Diagram — Post-Calibration')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Model Calibration — Eagle Ridge Thermal Scorer', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Save Artifacts

In [ ]:
# Save XGBoost model
model_path = ARTIFACT_PATH / 'thermal_scorer.json'
final_model.save_model(str(model_path))
print(f'Model saved to: {model_path}')

# Save feature importances
imp_path = ARTIFACT_PATH / 'feature_importances.json'
imp_df.to_json(str(imp_path), orient='records', indent=2)
print(f'Feature importances saved to: {imp_path}')

# Save calibration metadata
cal_meta = {
    'calibration_method': 'isotonic',
    'cv_folds': 5,
    'mean_auc': float(np.mean(auc_scores)),
    'mean_brier': float(np.mean(brier_scores)),
    'training_samples': len(y),
    'positive_class_fraction': float(y.mean()),
    'feature_names': feature_names,
    'label_threshold': 0.50,
    'note': 'Trained on synthetic data from MockWeatherProvider. Replace with real flight-derived labels for production.',
}
cal_path = ARTIFACT_PATH / 'model_metadata.json'
with open(cal_path, 'w') as f:
    json.dump(cal_meta, f, indent=2)
print(f'Metadata saved to: {cal_path}')

print('\nAll artifacts saved. To use in the backend:')
print('  scorer = ThermalScorer()')
print('  scorer.load_model("backend/ml/artifacts/thermal_scorer.json")')

## 6. Model Limitations

This baseline model has significant limitations:

1. **Synthetic labels**: The `y` labels are derived from the rule-based `ThermalScorer` formula itself. The XGBoost model is learning to approximate a formula it has indirect access to. This is useful for testing the pipeline but provides no new information.

2. **No real flight data**: Until IGC tracks with climb annotations are used as labels, this model will not improve on the rule-based baseline.

3. **Site specificity**: Once trained on Eagle Ridge flight data, this model should not be deployed at other sites without retraining.

4. **Temporal structure**: The synthetic data has a consistent diurnal pattern with low noise. Real weather and flight data will have more variance, requiring more sophisticated time-series cross-validation.

**Next steps:**
- Import real IGC files: `uv run python scripts/import_igc.py --dir /path/to/igc_files`
- Generate flight-derived labels using segment vario data
- Retrain with temporal (date-based) cross-validation
- Register the model: `uv run python scripts/train_baseline.py`